# Global E-Commerce Sales Data Cleaning
Synapse PySpark notebook for cleaning and transforming the dataset.

In [ ]:
# Synapse PySpark Notebook Code
# Global E-Commerce Sales Data Cleaning and Transformation

from pyspark.sql.functions import col, to_date, month, year

# Input path: cleaned/raw CSV location in ADLS Gen2
input_path = "abfss://shinyblobcontainer@shinyshastoreacc.dfs.core.windows.net/global E commerce sales dataset/*.csv"

# Read CSV file
df = spark.read.option("header", "true").option("inferSchema", "true").csv(input_path)

# Display initial information
print("Initial row count:", df.count())
print("Columns:", df.columns)

display(df.limit(10))

# Remove duplicate records
df_cleaned = df.dropDuplicates()

# Drop rows with null values for this project scope
df_cleaned = df_cleaned.dropna()

# Standardize date column if present
if "Transaction Date" in df_cleaned.columns:
    df_cleaned = df_cleaned.withColumn("Transaction_Date", to_date(col("Transaction Date")))
    df_cleaned = df_cleaned.withColumn("Transaction_Month", month(col("Transaction_Date")))
    df_cleaned = df_cleaned.withColumn("Transaction_Year", year(col("Transaction_Date")))

# Cast numeric columns if present
if "Price" in df_cleaned.columns:
    df_cleaned = df_cleaned.withColumn("Price", col("Price").cast("double"))
if "Quantity" in df_cleaned.columns:
    df_cleaned = df_cleaned.withColumn("Quantity", col("Quantity").cast("integer"))

# Create revenue column
if "Price" in df_cleaned.columns and "Quantity" in df_cleaned.columns:
    df_cleaned = df_cleaned.withColumn("Revenue", col("Price") * col("Quantity"))

print("Cleaned row count:", df_cleaned.count())
display(df_cleaned.limit(10))

# Output path: cleaned parquet location
output_path = "abfss://shinyblobcontainer@shinyshastoreacc.dfs.core.windows.net/cleaned/ecommerce_cleaned"
df_cleaned.write.mode("overwrite").parquet(output_path)

# Optional Power BI-friendly single parquet output
powerbi_output_path = "abfss://shinyblobcontainer@shinyshastoreacc.dfs.core.windows.net/gold/ecommerce_sales_powerbi"
df_cleaned.coalesce(1).write.mode("overwrite").parquet(powerbi_output_path)
